# Phase 3: Question Selection and Validation

## Tiller Restaurant Analytics Project

**Objective:** Evaluate all four candidate business questions and select the strongest one for detailed analysis.

**Date:** September 10, 2026

---

## 1. Context and Objective

### Background
Phase 2 identified **demand forecasting** as one of the strongest opportunities in the Tiller data because:
- Substantial historical transaction data across 21 stores
- Clear patterns in demand over time
- 5 years of data (Oct 2015 - Nov 2020)
- 1.28M orders to analyze

### The Challenge
Multiple candidate questions could address demand forecasting, but they have different:
- Business value propositions
- Analytical approaches
- Product implications
- Feasibility constraints

### Phase 3 Objective
Evaluate all four candidate questions **at equal depth** to make a defensible selection based on evidence, not assumptions.

---

## 2. The Four Candidate Questions

### Q1: Pattern Identification
**"Can Tiller identify patterns in store-level demand over time?"**
- Focus: Understanding demand patterns descriptively
- Value: Benchmarking, insights, anomaly detection
- Risk: May remain descriptive without clear actions

### Q2: Historical Forecasting
**"Can Tiller forecast weekly store-level demand 1–2 weeks ahead using historical patterns?"**
- Focus: Predicting future demand from historical patterns
- Value: Direct decision support (staffing, inventory)
- Risk: Forecasting accuracy may be limited

### Q3: Historical + Calendar Forecasting
**"Can Tiller accurately forecast upcoming transaction demand at store level using historical and calendar patterns?"**
- Focus: Forecasting with calendar features (week, month, season)
- Value: Calendar-aware forecasts (holidays, seasons)
- Risk: Calendar may add complexity without value

### Q4: AI/ML Forecasting
**"Can Tiller use AI/ML to forecast store-level demand?"**
- Focus: Testing ML approaches for forecasting
- Value: Potentially higher accuracy, automated feature discovery
- Risk: Technology-first (assumes ML is appropriate)

---

## 3. Evaluation Framework

Each question was evaluated against the same 10 criteria:

1. **Business relevance** - Does this solve a meaningful problem?
2. **Evidence available** - Does the dataset support this?
3. **Analytical signal** - Is there evidence the phenomenon exists?
4. **Analytical depth** - Can this produce meaningful analysis?
5. **Actionability** - Could a customer make better decisions?
6. **Product potential** - Could this become a Tiller feature?
7. **Advanced analytics opportunity** - Is ML/statistical modeling justified?
8. **Feasibility** - Can we answer this within project scope?
9. **Risks/ambiguity** - What could make this misleading?
10. **Overall assessment** - Strong / Moderate / Weak

---

## 4. Data Foundation

Before evaluating questions, we confirmed the data foundation:

In [ ]:
from google.cloud import bigquery
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

client = bigquery.Client(project="le-wagon-da-502302")

# Data foundation check
query = """
WITH store_stats AS (
  SELECT
    id_store,
    DATE_DIFF(MAX(DATE(date_opened)), MIN(DATE(date_opened)), DAY) AS tenure_days,
    COUNT(*) AS total_orders
  FROM `le-wagon-da-502302.tiller.order_data`
  GROUP BY id_store
)
SELECT
  'Total stores' AS metric,
  CAST(COUNT(*) AS STRING) AS value
FROM store_stats
UNION ALL
SELECT 'Stores with >1 year', CAST(SUM(CASE WHEN tenure_days >= 365 THEN 1 ELSE 0 END) AS STRING)
FROM store_stats
UNION ALL
SELECT 'Stores with >2 years', CAST(SUM(CASE WHEN tenure_days >= 730 THEN 1 ELSE 0 END) AS STRING)
FROM store_stats
UNION ALL
SELECT 'Total orders', CAST(SUM(total_orders) AS STRING)
FROM store_stats
"""

df_foundation = client.query(query).to_dataframe()
print("Data Foundation:")
df_foundation.to_string(index=False)

Data Foundation:


'              metric   value\n        Total stores      21\n Stores with >1 year      21\nStores with >2 years      19\n        Total orders 1281148'

**Result:** ✅ Strong foundation - 21 stores, 19 with >2 years history, 1.28M orders

## 5. Testing Analytical Signal

### 5.1 Store Variation (Q1, Q2, Q3, Q4)

Do stores have meaningful variation in demand?

In [ ]:
query_variation = """
WITH daily_agg AS (
  SELECT
    id_store,
    DATE(date_opened) AS order_date,
    COUNT(*) AS orders
  FROM `le-wagon-da-502302.tiller.order_data`
  GROUP BY id_store, DATE(date_opened)
),
store_profiles AS (
  SELECT
    id_store,
    AVG(orders) AS mean,
    ROUND(STDDEV(orders) / AVG(orders), 2) AS cv
  FROM daily_agg
  GROUP BY id_store
)
SELECT
  ROUND(MIN(mean), 1) AS min_mean,
  ROUND(MAX(mean), 1) AS max_mean,
  ROUND(MAX(mean) / MIN(mean), 1) AS max_min_ratio,
  ROUND(AVG(cv), 2) AS avg_cv
FROM store_profiles
"""

df_variation = client.query(query_variation).to_dataframe()
print("Store Variation:")
df_variation.to_string(index=False)
print(f"\nInterpretation: {df_variation['max_min_ratio'].values[0]:.1f}x difference between smallest and largest stores")

Store Variation:

Interpretation: 139.8x difference between smallest and largest stores


**Finding:** ✅ Strong variation (139.8x) - meaningful patterns exist between stores

### 5.2 Temporal Autocorrelation (Q1, Q2, Q3, Q4)

Does past demand predict future demand?

In [ ]:
query_temporal = """
WITH daily_agg AS (
  SELECT
    id_store,
    DATE(date_opened) AS order_date,
    COUNT(*) AS orders
  FROM `le-wagon-da-502302.tiller.order_data`
  GROUP BY id_store, DATE(date_opened)
),
with_lag AS (
  SELECT
    id_store,
    order_date,
    orders,
    LAG(orders, 1) OVER (PARTITION BY id_store ORDER BY order_date) AS lag_1d
  FROM daily_agg
),
correlations AS (
  SELECT
    id_store,
    ROUND(CORR(orders, lag_1d), 3) AS corr_1d
  FROM with_lag
  WHERE lag_1d IS NOT NULL
  GROUP BY id_store
)
SELECT
  ROUND(AVG(corr_1d), 3) AS avg_corr
FROM correlations
"""

df_temporal = client.query(query_temporal).to_dataframe()
print(f"Average temporal autocorrelation: {df_temporal['avg_corr'].values[0]:.3f}")
print("Interpretation: Moderate-strong temporal patterns exist")

Average temporal autocorrelation: 0.396
Interpretation: Moderate-strong temporal patterns exist


**Finding:** ✅ Temporal patterns exist (0.396 average correlation)

### 5.3 Day-of-Week Patterns (Q1, Q2, Q3)

Are there strong day-of-week patterns?

In [ ]:
query_dow = """
WITH daily_agg AS (
  SELECT
    id_store,
    DATE(date_opened) AS order_date,
    COUNT(*) AS orders
  FROM `le-wagon-da-502302.tiller.order_data`
  GROUP BY id_store, DATE(date_opened)
),
dow_agg AS (
  SELECT
    id_store,
    EXTRACT(DAYOFWEEK FROM order_date) AS dow,
    AVG(orders) AS avg_orders
  FROM daily_agg
  GROUP BY id_store, EXTRACT(DAYOFWEEK FROM order_date)
),
store_range AS (
  SELECT
    id_store,
    MAX(avg_orders) / MIN(avg_orders) AS dow_ratio
  FROM dow_agg
  GROUP BY id_store
)
SELECT
  ROUND(AVG(dow_ratio), 2) AS avg_ratio,
  ROUND(MAX(dow_ratio), 2) AS max_ratio
FROM store_range
"""

df_dow = client.query(query_dow).to_dataframe()
print(f"Average DOW ratio: {df_dow['avg_ratio'].values[0]:.2f}x")
print(f"Maximum DOW ratio: {df_dow['max_ratio'].values[0]:.2f}x")
print("Interpretation: Strong day-of-week patterns exist")

Average DOW ratio: 7.68x
Maximum DOW ratio: 33.43x
Interpretation: Strong day-of-week patterns exist


**Finding:** ✅ Very strong DOW patterns (7.68x average ratio)

### 5.4 Weekly Forecasting Viability (Q2, Q3, Q4)

Is weekly forecasting viable across all stores?

In [ ]:
query_weekly = """
WITH weekly_agg AS (
  SELECT
    id_store,
    DATE_TRUNC(DATE(date_opened), WEEK) AS week,
    COUNT(*) AS orders
  FROM `le-wagon-da-502302.tiller.order_data`
  GROUP BY 1, 2
),
with_lag AS (
  SELECT
    id_store,
    week,
    orders,
    LAG(orders, 1) OVER (PARTITION BY id_store ORDER BY week) AS lag_1w
  FROM weekly_agg
),
store_performance AS (
  SELECT
    id_store,
    ROUND(CORR(orders, lag_1w), 3) AS lag_corr,
    ROUND(AVG(orders), 1) AS avg_orders,
    ROUND(STDDEV(orders) / AVG(orders), 2) AS cv,
    COUNT(*) AS weeks
  FROM with_lag
  WHERE lag_1w IS NOT NULL
  GROUP BY 1
)
SELECT * FROM store_performance
ORDER BY avg_orders DESC
"""

df_weekly = client.query(query_weekly).to_dataframe()
print("Weekly forecasting viability by store:")
print(df_weekly.to_string())
print(f"\nMedian lag correlation: {df_weekly['lag_corr'].median():.3f}")
print(f"Stores with corr > 0.3: {(df_weekly['lag_corr'] > 0.3).sum()} / {len(df_weekly)}")

Weekly forecasting viability by store:
    id_store  lag_corr  avg_orders    cv  weeks
0       4151     0.919      5841.3  0.54    149
1       5281     0.851       903.1  0.33    141
2       8291     0.797       648.1  0.30     83
3       5617     0.626       175.8  0.63     97
4       6008     0.095       159.5  0.32    116
5       2035     0.705       149.8  0.32    204
6       4803     0.701       140.2  0.36    128
7       6827     0.655       130.7  0.33    113
8       5210     0.380        98.4  0.26    133
9       1513     0.801        97.7  0.29    232
10      4364     0.618        88.2  0.39    149
11      7786     0.651        77.0  0.45     82
12       351     0.520        71.5  0.25    205
13      5498     0.555        70.3  0.50    132
14      6830     0.674        66.6  0.35    115
15       360     0.519        58.3  0.29    209
16      7872     0.753        58.3  0.38     92
17      5860     0.336        51.8  0.31     87
18      7304     0.445        51.6  0.30     97
1

**Finding:** ✅ Strong weekly signal (0.651 median correlation, 20/21 stores > 0.3)

## 6. Question Evaluation Results

### Q1: Pattern Identification

**Strengths:**
- ✅ Strong patterns exist (139.8x variation, 7.68x DOW)
- ✅ Clear business relevance (understanding demand)
- ✅ Feasible with existing data

**Weaknesses:**
- ❌ Actionability gap (patterns → decisions needs translation)
- ❌ Less analytical depth (primarily descriptive)
- ❌ May be "interesting" without being "actionable"

**Verdict:** MODERATE-HIGH - Keep as supporting analysis

### Q2: Historical Forecasting

**Strengths:**
- ✅ Strongest analytical signal (0.651 correlation)
- ✅ Clearest business value (forecasting → direct decisions)
- ✅ High actionability (no translation gap)
- ✅ Strong product potential (forecasting tool)
- ✅ Feasible within scope

**Weaknesses:**
- ⚠️ 22.8% WAPE may limit improvement room
- ⚠️ Store 4151 dominance complicates evaluation

**Verdict:** HIGH - **SELECTED**

### Q3: Historical + Calendar Forecasting

**Strengths:**
- ✅ Same business value as Q2 (forecasting)
- ✅ Calendar features easy to test

**Weaknesses:**
- ❌ Calendar variables show weak signal (0.056 correlation)
- ❌ Historical already captures most signal
- ❌ Adds complexity without proven value

**Verdict:** MODERATE-HIGH - Test calendar as hypothesis within Q2

### Q4: AI/ML Forecasting

**Strengths:**
- ✅ Problem complexity justifies testing ML
- ✅ Feature richness available
- ✅ High analytical depth

**Weaknesses:**
- ❌ Technology-first framing (assumes ML appropriate)
- ❌ Must test against baselines (don't assume ML wins)
- ❌ Portfolio-driven selection risk

**Verdict:** MODERATE-HIGH - Test ML as hypothesis within Q2

## 7. Head-to-Head Comparison

| Criterion | Q1 Patterns | Q2 Historical | Q3 Hist+Cal | Q4 AI/ML |
|-----------|-------------|---------------|-------------|----------|
| Business relevance | MOD-HIGH | **HIGH** | HIGH | HIGH |
| Evidence | STRONG | **STRONG** | MOD-STRONG | STRONG |
| Analytical signal | STRONG | **STRONG** | MODERATE | MOD-HIGH |
| Actionability | MODERATE | **HIGH** | HIGH | HIGH |
| Product potential | MODERATE | **HIGH** | MOD-HIGH | MOD-HIGH |
| Feasibility | STRONG | **STRONG** | STRONG | MOD-STRONG |
| **OVERALL** | MOD-HIGH | **HIGH** | MOD-HIGH | MOD-HIGH |

## 8. Final Decision

### Selected Question

**Q2: "Can Tiller forecast weekly store-level demand 1–2 weeks ahead using historical patterns?"**

### Top 3 Reasons

1. **Strongest analytical signal**
   - 0.651 median weekly lag correlation
   - 20/21 stores > 0.3 correlation
   - 40% improvement from weekly aggregation

2. **Clearest business value**
   - Forecasting → direct decisions (staffing, inventory)
   - Weekly horizons match business planning cycles
   - Clear ROI story (reduce overstaffing, avoid stockouts)

3. **Best balance of depth + feasibility**
   - High analytical depth (forecasting + feature engineering + ML testing)
   - Feasible within project scope (21 stores, weekly forecasts)
   - Manageable timeline (2-3 weeks)

### Why Not The Others

- **Q1:** Actionability gap (patterns need translation to decisions)
- **Q3:** Calendar adds complexity without proven value (0.056 correlation)
- **Q4:** ML should be tested as hypothesis, not assumed as question

## 9. Key Evidence Summary

### Data Foundation
- 21 stores, 1.28M orders, 5 years history
- 19 stores with >2 years data
- All stores CV < 1.0 (stable dispersion)

### Forecasting Baseline
- Weekly WAPE: 22.8%
- Daily WAPE: 40.0%
- Weekly improvement: 40%
- Weekly lag correlation: 0.651 median

### Store Insights
- Store 4151: 68% volume, 0.919 correlation (report separately)
- High-error stores: 5617, 7786, 5498 (unexplained)
- Systematic missingness: 17 stores >20% missing weekdays

### Calendar Signal
- Day of week: 7.68x ratio (strong)
- Week of year: 0.056 correlation (weak)
- **Conclusion:** Test calendar as hypothesis, don't assume value

## 10. Next Steps

### Phase 4: Hypothesis Testing and Forecasting Analysis

**Hypotheses to test within Q2:**
1. Recent demand predicts upcoming demand (lag features)
2. Multiple historical periods improve forecasts (rolling averages)
3. Store characteristics affect forecastability
4. Calendar features add predictive value (test, don't assume)
5. ML provides meaningful incremental improvement (test fairly)

**Approach:**
- Start with simple baseline (naive forecast)
- Add historical features incrementally
- Test calendar features as hypothesis
- Test ML only if simpler approaches insufficient
- Compare all approaches fairly

**Success criteria:**
- Beat 22.8% WAPE meaningfully (target <20%)
- Improve high-error stores (5617, 7786, 5498)
- Consistent across stores (median store WAPE <25%)
- ML improvement >5% (justifies complexity)

---

## Conclusion

Phase 3 evaluated four candidate questions at equal depth using a comprehensive framework.

**Q2 (Historical Forecasting)** was selected because it has:
- The strongest analytical evidence (0.651 correlation)
- The clearest business value (forecasting → decisions)
- The best balance of depth and feasibility

Q3 and Q4 become **hypotheses within Q2** rather than separate questions:
- Test calendar features (but don't expect much value based on 0.056 correlation)
- Test ML (but don't assume it wins - must demonstrate improvement)

This keeps the investigation **evidence-driven** rather than method-driven.

---

**End of Phase 3: Question Selection and Validation**